# 豆瓣 Top250 数据写入 MySQL

这个 Notebook 用来把 `output/douban_top250_movies_list.csv` 中的电影数据写入你已经创建好的 MySQL 表。

默认目标表：`douban_top250_movies`

In [ ]:
# %pip install pandas pymysql openpyxl

In [1]:
from pathlib import Path

import pandas as pd
import pymysql

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

In [2]:
CSV_PATH = PROJECT_ROOT / 'output' / 'douban_top250_movies_list.csv'

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': '2285798898',
    'database': 'xiaoqi',
    'charset': 'utf8mb4'
}

TABLE_NAME = 'douban_top250_movies'

## 1. 读取 CSV

In [3]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f'找不到 CSV 文件: {CSV_PATH.resolve()}')

df_raw = pd.read_csv(CSV_PATH)
print(f'原始记录数: {len(df_raw)}')
df_raw.head()

原始记录数: 250


,rank,title_cn,title_other,rating,rating_count,quote,detail_url,poster_url,crawl_time
0,1,肖申克的救赎,The Shawshank Redemption / 月黑高飞(港) / 刺激1995(台),9.7,3287797,希望让人自由。,https://movie.douban.com/subject/1292052/,https://img3.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
1,2,霸王别姬,再见，我的妾 / Farewell My Concubine,9.6,2425821,风华绝代。,https://movie.douban.com/subject/1291546/,https://img1.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
2,3,泰坦尼克号,Titanic / 铁达尼号(港 / 台),9.5,2501117,失去的才是永恒的。,https://movie.douban.com/subject/1292722/,https://img9.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
3,4,阿甘正传,Forrest Gump / 福雷斯特·冈普,9.5,2432641,一部美国近现代史。,https://movie.douban.com/subject/1292720/,https://img3.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
4,5,千与千寻,千と千尋の神隠し / 神隐少女(台) / 千与千寻的神隐,9.4,2539568,最好的宫崎骏，最好的久石让。,https://movie.douban.com/subject/1291561/,https://img1.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44


## 2. 清洗字段并映射到数据库列

In [4]:
expected_columns = [
    'rank',
    'title_cn',
    'title_other',
    'rating',
    'rating_count',
    'quote',
    'detail_url',
    'poster_url',
    'crawl_time'
]

missing_columns = [column for column in expected_columns if column not in df_raw.columns]
if missing_columns:
    raise ValueError(f'CSV 缺少必要字段: {missing_columns}')

df_movies = df_raw[expected_columns].copy()
df_movies = df_movies.rename(
    columns={
        'rank': 'movie_rank',
        'quote': 'quote_text'
    }
)

df_movies['movie_rank'] = pd.to_numeric(df_movies['movie_rank'], errors='coerce').astype('Int64')
df_movies['rating'] = pd.to_numeric(df_movies['rating'], errors='coerce')
df_movies['rating_count'] = pd.to_numeric(df_movies['rating_count'], errors='coerce').astype('Int64')
df_movies['crawl_time'] = pd.to_datetime(df_movies['crawl_time'], errors='coerce')

df_movies = df_movies.dropna(subset=['movie_rank', 'title_cn'])
df_movies = df_movies.drop_duplicates(subset=['detail_url'], keep='last')
df_movies = df_movies.sort_values(by='movie_rank').reset_index(drop=True)

print(f'清洗后记录数: {len(df_movies)}')
df_movies.head()

清洗后记录数: 250


,movie_rank,title_cn,title_other,rating,rating_count,quote_text,detail_url,poster_url,crawl_time
0,1,肖申克的救赎,The Shawshank Redemption / 月黑高飞(港) / 刺激1995(台),9.7,3287797,希望让人自由。,https://movie.douban.com/subject/1292052/,https://img3.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
1,2,霸王别姬,再见，我的妾 / Farewell My Concubine,9.6,2425821,风华绝代。,https://movie.douban.com/subject/1291546/,https://img1.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
2,3,泰坦尼克号,Titanic / 铁达尼号(港 / 台),9.5,2501117,失去的才是永恒的。,https://movie.douban.com/subject/1292722/,https://img9.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
3,4,阿甘正传,Forrest Gump / 福雷斯特·冈普,9.5,2432641,一部美国近现代史。,https://movie.douban.com/subject/1292720/,https://img3.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44
4,5,千与千寻,千と千尋の神隠し / 神隐少女(台) / 千与千寻的神隐,9.4,2539568,最好的宫崎骏，最好的久石让。,https://movie.douban.com/subject/1291561/,https://img1.doubanio.com/view/photo/s_ratio_p...,2026-05-19 23:47:44


## 3. 测试数据库连接

In [5]:
connection = pymysql.connect(**DB_CONFIG)
print('MySQL 连接成功')
connection.close()

MySQL 连接成功


## 4. 写入数据库

这里使用 `ON DUPLICATE KEY UPDATE`，所以如果 `movie_rank` 或 `detail_url` 已存在，会更新原记录。

In [6]:
def value_or_none(value):
    if pd.isna(value):
        return None
    if isinstance(value, str) and not value.strip():
        return None
    return value


insert_sql = f"""
    INSERT INTO `{TABLE_NAME}` (
        movie_rank,
        title_cn,
        title_other,
        rating,
        rating_count,
        quote_text,
        detail_url,
        poster_url,
        crawl_time
    ) VALUES (
        %s, %s, %s, %s, %s, %s, %s, %s, %s
    )
    ON DUPLICATE KEY UPDATE
        movie_rank = VALUES(movie_rank),
        title_cn = VALUES(title_cn),
        title_other = VALUES(title_other),
        rating = VALUES(rating),
        rating_count = VALUES(rating_count),
        quote_text = VALUES(quote_text),
        poster_url = VALUES(poster_url),
        crawl_time = VALUES(crawl_time)
"""

rows = []
for row in df_movies.itertuples(index=False):
    rows.append(
        (
            int(row.movie_rank) if pd.notna(row.movie_rank) else None,
            row.title_cn,
            value_or_none(row.title_other),
            float(row.rating) if pd.notna(row.rating) else None,
            int(row.rating_count) if pd.notna(row.rating_count) else None,
            value_or_none(row.quote_text),
            value_or_none(row.detail_url),
            value_or_none(row.poster_url),
            row.crawl_time.to_pydatetime() if pd.notna(row.crawl_time) else None,
        )
    )

print(f'准备写入 {len(rows)} 条记录')

准备写入 250 条记录


In [7]:
connection = pymysql.connect(**DB_CONFIG)

try:
    with connection.cursor() as cursor:
        cursor.executemany(insert_sql, rows)
    connection.commit()
    print(f'写入完成，共处理 {len(rows)} 条记录')
finally:
    connection.close()

写入完成，共处理 250 条记录


## 5. 验证写入结果

In [8]:
check_sql = f"""
    SELECT movie_rank, title_cn, rating, rating_count, crawl_time
    FROM `{TABLE_NAME}`
    ORDER BY movie_rank ASC
    LIMIT 10
"""

count_sql = f"SELECT COUNT(*) AS total_count FROM `{TABLE_NAME}`"

connection = pymysql.connect(**DB_CONFIG)
try:
    df_preview = pd.read_sql(check_sql, connection)
    df_count = pd.read_sql(count_sql, connection)
finally:
    connection.close()

print(df_count)
df_preview

   total_count
0          250


C:\Users\22857\AppData\Local\Temp\ipykernel_46304\2797491098.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_preview = pd.read_sql(check_sql, connection)
C:\Users\22857\AppData\Local\Temp\ipykernel_46304\2797491098.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql(count_sql, connection)


,movie_rank,title_cn,rating,rating_count,crawl_time
0,1,肖申克的救赎,9.7,3287797,2026-05-19 23:47:44
1,2,霸王别姬,9.6,2425821,2026-05-19 23:47:44
2,3,泰坦尼克号,9.5,2501117,2026-05-19 23:47:44
3,4,阿甘正传,9.5,2432641,2026-05-19 23:47:44
4,5,千与千寻,9.4,2539568,2026-05-19 23:47:44
5,6,美丽人生,9.5,1483650,2026-05-19 23:47:44
6,7,星际穿越,9.4,2186810,2026-05-19 23:47:44
7,8,这个杀手不太冷,9.4,2554334,2026-05-19 23:47:44
8,9,盗梦空间,9.4,2327663,2026-05-19 23:47:44
9,10,楚门的世界,9.4,2021756,2026-05-19 23:47:44
